# Phase 1: propagation-tree visualization

This notebook visualizes the propagation-tree artifacts generated by `explore_data.ipynb`. It uses the same central `config.py` paths as the rest of the project, reads the saved Phase 1 summary tables from the configured evaluation directory, reloads the selected tree files, and produces visual checks of how true and false cascades differ structurally.

The work carried out here is:

1. Load the configured evaluation directory and Phase 1 catalog files.
2. Reconstruct selected propagation trees from their saved graph paths.
3. Plot 3 to 5 representative true and false cascades with root nodes highlighted.
4. Plot depth profiles to show how each cascade expands over time or levels.
5. Compare aggregate tree statistics such as cascade size, maximum depth, maximum width, and branching ratio.
6. Save figures under the configured evaluation directory so they can be reused in reports and later phases.

The goal is not to train a model yet. The goal is to visually verify that the propagation data are parsed correctly and that the tree-level features needed for later graph modeling are meaningful.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import re
import math
import random
import sys

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Import the central project configuration. The /mnt/data fallback is useful
# when running this notebook inside the ChatGPT sandbox.
for candidate in [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]:
    if (candidate / 'config.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from config import cfg, ROOT as CONFIG_ROOT
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import config.py. Place config.py in the project root or start Jupyter from the project root.'
    ) from exc

RANDOM_SEED = cfg.seed
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path(CONFIG_ROOT).resolve()
EVAL_DIR = cfg.paths.evaluation
SUMMARY_PATH = EVAL_DIR / 'propagation_tree_summary.csv'
SAMPLES_PATH = EVAL_DIR / 'sample_tree_ids.csv'
FIG_DIR = EVAL_DIR / 'tree_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Evaluation directory: {EVAL_DIR}')
print(f'Figure directory: {FIG_DIR}')


## 1. Load the Phase 1 catalog from the configured evaluation directory

Run `explore_data.ipynb` first. It creates the summary and sample CSV files used here under `cfg.paths.evaluation`.


In [ ]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'Missing {SUMMARY_PATH}. Run explore_data.ipynb first.')

tree_summary = pd.read_csv(SUMMARY_PATH, dtype={'tweet_id': str, 'content_id': str})
if SAMPLES_PATH.exists():
    sample_trees = pd.read_csv(SAMPLES_PATH, dtype={'tweet_id': str, 'content_id': str})
else:
    sample_trees = tree_summary.sort_values(['cascade_size','max_depth'], ascending=False).head(6)

print('Summary rows:', len(tree_summary))
print('Sample rows:', len(sample_trees))
display(sample_trees[['dataset','tweet_id','label','cascade_size','max_depth','max_width','graph_path']])

## 2. Tree parsing helpers

These are repeated so this notebook can reload graph files directly from the paths saved by the exploration notebook.

In [ ]:
def normalize_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s or s.lower() in {'nan', 'none', 'null'}:
        return None
    if re.fullmatch(r'\d+\.0', s):
        s = s[:-2]
    return s


def parse_edge_line(line):
    """Parse one edge line from a Twitter15/16 tree file.

    Expected format:
        ['parent_uid', 'parent_tweet_id', 'delay_min']->['child_uid', 'child_tweet_id', 'delay_min']

    Root sentinel:
        ['ROOT', 'ROOT', '0.0']->['uid', 'tweet_id', '0.0']

    Returns (parent_uid, child_uid, delay_float) or None.
    """
    raw = line.strip()
    if not raw or '->' not in raw:
        return None
    halves = raw.split('->', 1)
    if len(halves) != 2:
        return None
    def extract_fields(s):
        return re.findall(r"'([^']*?)'", s)
    parent_fields = extract_fields(halves[0])
    child_fields  = extract_fields(halves[1])
    if len(parent_fields) < 1 or len(child_fields) < 1:
        return None
    parent_uid = normalize_id(parent_fields[0])
    child_uid  = normalize_id(child_fields[0])
    delay = None
    if len(child_fields) >= 3:
        try:
            delay = float(child_fields[2])
        except ValueError:
            pass
    return parent_uid, child_uid, delay


def graph_from_edges(edge_rows, root_id=None):
    """Build a DiGraph from (parent_uid, child_uid, delay) tuples.

    Rows where parent_uid matches the root sentinel are re-rooted to root_id.
    """
    ROOT_SENTINEL = cfg.twitter15.root_sentinel  # 'ROOT'
    G = nx.DiGraph()
    root_id = normalize_id(root_id)
    for parent, child, delay in edge_rows:
        parent = normalize_id(parent)
        child  = normalize_id(child)
        if parent is None or child is None:
            continue
        if parent.upper() == ROOT_SENTINEL.upper() and root_id:
            parent = root_id
        G.add_edge(parent, child)
        if delay is not None:
            G.nodes[child]['delay'] = delay
    if root_id:
        G.add_node(root_id)
    return G


def load_graph_for_row(row):
    """Reconstruct a DiGraph from a catalog row.

    graph_path may be:
      - A .txt file  : Twitter15/16 tree file
      - A directory  : WICO Graph per-tweet folder containing
                       cfg.wico.graph_edges_file ('edges.txt')
    """
    path    = Path(row['graph_path'])
    root_id = normalize_id(row.get('content_id', row.get('tweet_id')))
    if not path.exists():
        raise FileNotFoundError(path)

    # ── WICO Graph: graph_path is a directory ─────────────────────────────
    if path.is_dir():
        edges_path = path / cfg.wico.graph_edges_file
        G = nx.DiGraph()
        if edges_path.exists():
            for line in edges_path.read_text(errors='ignore').splitlines():
                parts = line.strip().split()
                if len(parts) == 2:
                    src, dst = normalize_id(parts[0]), normalize_id(parts[1])
                    if src and dst:
                        G.add_edge(src, dst)
        nodes_path = path / cfg.wico.graph_nodes_file
        if nodes_path.exists():
            nodes_df = pd.read_csv(nodes_path, dtype=str)
            nodes_df.columns = [c.strip().lower() for c in nodes_df.columns]
            for _, nr in nodes_df.iterrows():
                nid = normalize_id(nr.get('id'))
                if nid:
                    G.add_node(nid)
                    for feat in ('time', 'friends', 'followers'):
                        try:
                            G.nodes[nid][feat] = int(nr[feat])
                        except (ValueError, KeyError, TypeError):
                            pass
        if root_id:
            G.add_node(root_id)
        return G

    # ── Twitter15/16: graph_path is a .txt tree file ──────────────────────
    edge_rows = []
    for line in path.read_text(errors='ignore').splitlines():
        parsed = parse_edge_line(line)
        if parsed:
            edge_rows.append(parsed)
    return graph_from_edges(edge_rows, root_id=root_id)


## 3. Layout and plotting helpers

The plots highlight the root node and show the outward reshare structure. For very large cascades, only the first few hundred nodes are drawn to keep the visualization readable.

In [ ]:
def find_root(G, preferred=None):
    preferred = normalize_id(preferred)
    if preferred in G:
        return preferred
    roots = [n for n in G.nodes if G.in_degree(n) == 0]
    return roots[0] if roots else (next(iter(G.nodes)) if G.number_of_nodes() else None)


def depth_subgraph(G, root, max_nodes=250):
    if root is None or root not in G:
        return G.copy(), {}
    depths = nx.single_source_shortest_path_length(G, root)
    nodes = sorted(depths, key=lambda n: (depths[n], str(n)))[:max_nodes]
    H = G.subgraph(nodes).copy()
    return H, {n: depths[n] for n in H.nodes}


def hierarchy_positions(G, root=None):
    root = find_root(G, root)
    if root is None:
        return {}
    depths = nx.single_source_shortest_path_length(G, root)
    by_depth = defaultdict(list)
    for n, d in depths.items():
        by_depth[d].append(n)
    pos = {}
    for d, nodes in by_depth.items():
        nodes = sorted(nodes, key=str)
        width = max(1, len(nodes) - 1)
        for i, n in enumerate(nodes):
            x = i / width - 0.5 if width else 0
            y = -d
            pos[n] = (x, y)
    # Add disconnected nodes if any.
    missing = [n for n in G.nodes if n not in pos]
    for i, n in enumerate(missing):
        pos[n] = (0.7 + (i % 10) * 0.05, -i // 10)
    return pos


def plot_tree(row, max_nodes=250):
    G = load_graph_for_row(row)
    root = find_root(G, row.get('content_id', row.get('tweet_id')))
    H, depths = depth_subgraph(G, root, max_nodes=max_nodes)
    pos = hierarchy_positions(H, root)
    plt.figure(figsize=(10, 7))
    sizes = [220 if n == root else 35 for n in H.nodes]
    nx.draw_networkx_edges(H, pos, arrows=True, arrowsize=8, alpha=0.35, width=0.8)
    nx.draw_networkx_nodes(H, pos, node_size=sizes, alpha=0.85)
    if root in H:
        nx.draw_networkx_nodes(H, pos, nodelist=[root], node_size=320, alpha=1.0)
    title = f"{row['dataset']} | {row.get('label','unknown')} | id={row.get('tweet_id')} | nodes={G.number_of_nodes()} edges={G.number_of_edges()}"
    if H.number_of_nodes() < G.number_of_nodes():
        title += f" | drawn first {H.number_of_nodes()} nodes"
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    safe_id = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(row.get('tweet_id')))
    out = FIG_DIR / f"tree_{row['dataset']}_{row.get('label','unknown')}_{safe_id}.png"
    plt.savefig(out, dpi=180)
    plt.show()
    print('Saved:', out)
    return G


def plot_depth_profile(G, title, root=None):
    root = find_root(G, root)
    if root is None:
        return
    depths = nx.single_source_shortest_path_length(G, root)
    counts = Counter(depths.values())
    xs = sorted(counts)
    ys = [counts[x] for x in xs]
    plt.figure(figsize=(7, 4))
    plt.bar(xs, ys)
    plt.xlabel('Depth from root')
    plt.ylabel('Number of nodes')
    plt.title(title)
    plt.tight_layout()
    safe_title = re.sub(r'[^A-Za-z0-9_.-]+', '_', title)[:120]
    out = FIG_DIR / f"depth_{safe_title}.png"
    plt.savefig(out, dpi=180)
    plt.show()
    print('Saved:', out)

## 4. Visualize 3 to 5 true and false example trees

The sample file usually contains three true and three false cascades. If one class is missing, the notebook visualizes the available labeled or largest cascades.

In [ ]:
visualized_graphs = []
for _, row in sample_trees.head(10).iterrows():
    if pd.isna(row.get('graph_path')):
        continue
    try:
        G = plot_tree(row, max_nodes=250)
        visualized_graphs.append((row, G))
    except Exception as e:
        print(f"Could not visualize {row.get('dataset')} {row.get('tweet_id')}: {e}")

print(f'Visualized {len(visualized_graphs)} cascades.')

## 5. Depth profiles for the same examples

Depth profiles make shallow-broad versus deep-chain propagation patterns easier to compare.

In [ ]:
for row, G in visualized_graphs:
    title = f"{row['dataset']} {row.get('label','unknown')} {row.get('tweet_id')} depth profile"
    plot_depth_profile(G, title, root=row.get('content_id', row.get('tweet_id')))

## 6. Aggregate true vs false comparisons

These are exploratory plots for Phase 1. They are not model results, but they help verify whether the available data contains structural differences that later notebooks can use.

In [ ]:
plot_df = tree_summary[tree_summary['label'].isin(['true','false'])].copy()
if plot_df.empty:
    print('No true/false labels available for aggregate comparison.')
else:
    for metric in ['cascade_size', 'max_depth', 'max_width', 'branching_ratio']:
        plt.figure(figsize=(8, 4))
        labels = ['true', 'false']
        data = [plot_df.loc[plot_df['label'].eq(label), metric].dropna().values for label in labels]
        plt.boxplot(data, labels=labels)
        plt.ylabel(metric)
        plt.title(f'{metric}: true vs false cascades')
        plt.tight_layout()
        out = FIG_DIR / f'aggregate_{metric}_true_vs_false.png'
        plt.savefig(out, dpi=180)
        plt.show()
        print('Saved:', out)

## 7. Dataset-specific comparison, including WICO

When WICO Text and WICO Graph are both present and joined, WICO appears here as its own dataset. This allows the notebook to keep Twitter15/Twitter16 exploration separate from the final WICO evaluation source.

In [ ]:
if tree_summary.empty:
    print('No tree summary available.')
else:
    metrics = ['cascade_size', 'max_depth', 'max_width', 'branching_ratio']
    grouped = tree_summary.groupby(['dataset','label'])[metrics].agg(['count','mean','median','max']).round(2)
    display(grouped)

    for dataset, part in tree_summary.groupby('dataset'):
        part = part[part['label'].isin(['true','false'])]
        if part.empty:
            continue
        plt.figure(figsize=(8, 4))
        data = [part.loc[part['label'].eq(label), 'cascade_size'].dropna().values for label in ['true','false']]
        plt.boxplot(data, labels=['true','false'])
        plt.ylabel('cascade_size')
        plt.title(f'{dataset}: cascade size by label')
        plt.tight_layout()
        out = FIG_DIR / f'{dataset}_cascade_size_by_label.png'
        plt.savefig(out, dpi=180)
        plt.show()
        print('Saved:', out)

## 8. Phase 1 outputs

Generated files:

- `evaluation/propagation_tree_summary.csv`
- `evaluation/sample_tree_ids.csv`
- `evaluation/tree_figures/*.png`

The next phase can consume the summary CSV for feature engineering and the graph files for PyG conversion.